In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms,datasets
import wandb
from key import KEY
import torch.nn.functional as F
import os
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

In [2]:
img = Image.open('data/img.jpg')

In [3]:
wandb.login(key=KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\최연호\_netrc
wandb: Currently logged in as: chldusgh0497 (chldusgh0497-jeonbuk) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
def data_load(image_size=64, batch_size=128):
    train_dir = 'data/archive/training_set/training_set'
    val_dir = 'data/archive/validation_set/validation_set'
    test_dir = 'data/archive/test_set/test_set'
    trans= transforms.Compose([transforms.Resize((image_size, image_size)), 
                           transforms.ToTensor(), 
                           transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])])


    train_dataset=datasets.ImageFolder(root=train_dir, transform=trans)
    val_dataset=datasets.ImageFolder(root=val_dir, transform=trans)
    test_dataset=datasets.ImageFolder(root=test_dir, transform=trans)
    train_loader= DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader= DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader= DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    return train_loader, val_loader, test_loader

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
    
    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.pool(x)
        return x

class convolution(nn.Module):
    def __init__(self, hidden_size_list, conv_layer_num, block_num, fc2_out, input_size):
        super(convolution, self).__init__()
        self.conv_layer_num = conv_layer_num
        self.block_num = block_num
        self.input_size = input_size
        
        # Block들을 동적으로 생성
        self.blocks = nn.ModuleList()
        
        # 각 Block당 Conv 레이어 수 계산
        conv_per_block = conv_layer_num // block_num
        
        in_channels = 3
        for block_idx in range(block_num):
            block_layers = nn.ModuleList()
            
            # 각 Block 내부의 Conv 레이어들
            for conv_idx in range(conv_per_block):
                global_conv_idx = block_idx * conv_per_block + conv_idx
                out_channels = hidden_size_list[global_conv_idx]
                block_layers.append(nn.Conv2d(in_channels, out_channels, 3, padding=1))
                in_channels = out_channels
            
            # Block 끝에 Pooling 추가
            block_layers.append(nn.MaxPool2d(2, 2))
            self.blocks.append(block_layers)
        
        # FC 레이어 크기 계산
        self.fc1_size = hidden_size_list[conv_layer_num]
        self.fc2_size = fc2_out
        
        # FC 레이어는 forward에서 동적으로 생성
        self.fc1 = None
        self.fc2 = nn.Linear(self.fc1_size, fc2_out)
    
    def forward(self, x):
        # Block들을 통과
        for block in self.blocks:
            for layer in block:
                if isinstance(layer, nn.Conv2d):
                    x = F.relu(layer(x))
                else:  # MaxPool2d
                    x = layer(x)
        
        # Flatten
        batch_size = x.size(0)
        features = x.size(1) * x.size(2) * x.size(3)
        x = x.view(batch_size, -1)
        
        # FC1 레이어 동적 생성
        if self.fc1 is None:
            self.fc1 = nn.Linear(features, self.fc1_size).to(x.device)
        
        # FC 레이어들 통과
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [6]:
def train_model(model,device, train_loader, val_loader, criterion, optimizer, epochs=10):
    


    for epoch in range(epochs):
        print(f'Starting Epoch: {epoch+1}...')
        running_loss = 0.0
        model.train()
        model.to(device)

        for i, data in enumerate(train_loader, 0):
            inputs, labels = data
            
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
        

        model.eval()
        with torch.no_grad():
            for i, data in enumerate(val_loader, 0):
                inputs, labels = data
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
    return model


In [7]:
def evaluate_model(model,device, test_loader):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            # 예측값과 실제값 저장
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    conf_matrix = confusion_matrix(all_labels, all_preds)

    return accuracy, conf_matrix

In [11]:
def process_model(learning_rate=0.001, image_size=64, batch_size=128, hidden_size_list=[32,64,128], block_num=2, conv_layer_num=2, epochs=10, i=0):
    def log_confusion_matrix(conf_matrix, class_names=['Cat', 'Dog']):
        data = []
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                data.append([class_names[i], class_names[j], conf_matrix[i][j]])

        table = wandb.Table(columns=["Actual", "Predicted", "Count"], data=data)
        wandb.log({"confusion_matrix_table": table})
    #데이터 호출
    last=hidden_size_list[-1]
    if block_num != 1:
        hidden_size_list.append(last*2)
        hidden_size_list.append(last*4)
    print(f'block_num: {block_num}')
    print(f'conv_layer_num: {conv_layer_num}')
    print(f'hidden_size_list: {hidden_size_list}')
    wandb.init(project="cnn_test_img_265", name=f"cnn_{i}_{block_num}_{conv_layer_num}",config={
        "model": "convolution",
        "learning_rate": learning_rate, 
        "image_size": image_size,
        "batch_size": batch_size, 
        "epochs": epochs,
        "hidden_size_list": hidden_size_list,
        "block_num": block_num,
        "conv_layer_num": conv_layer_num,
        })
    device= torch.device("cuda" if torch.cuda.is_available() else "cpu")


    train_loader, val_loader, test_loader= data_load(image_size, batch_size)

    model= convolution(hidden_size_list=hidden_size_list,conv_layer_num=conv_layer_num, block_num=block_num, fc2_out=2, input_size=image_size)
    #optimizer 선택
    optimizer= optim.Adam(model.parameters(), lr=learning_rate)
    criterion= nn.CrossEntropyLoss()

    #모델 생성
    #모델 학습
    model= train_model(model=model,device=device, train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, epochs=epochs)
    
    #모델평가
    accuracy, conf_matrix= evaluate_model(model=model, device=device, test_loader=test_loader)
    log_confusion_matrix(conf_matrix)
    wandb.log({"accuracy": accuracy})
    wandb.finish()

In [9]:
# params = {
#     "learning_rate": [0.001, 0.005, 0.01],
#     "image_size": [64, 128],
#     "batch_size": [128, 256],
#     "epochs": [ 20, 30],
#     "hidden_sizes": [[32,64,128],[64,128,256],[128,256,512]],
# }
params = {
    "learning_rate": [0.001],
    "image_size": [128],
    "batch_size": [64],
    "epochs": [20],
    "hidden_sizes": [[64,128,256]],
    "block_num": [2],
    "conv_layer_num": [2],
    
}
#hidden_sze가 계속 증가하는 이유 :  더 민감하게 반응하도록 하기 위해.


In [12]:

for learning_rate in params['learning_rate']:
    for image_size in params['image_size']:
        for batch_size in params['batch_size']:
            for epochs in params['epochs']:
                for hidden_size_list in params['hidden_sizes']:
                    for block_num in params['block_num']:
                        for conv_layer_num in params['conv_layer_num']:
                            process_model(learning_rate=learning_rate, image_size=image_size, batch_size=batch_size, epochs=epochs, hidden_size_list=hidden_size_list, block_num=block_num, conv_layer_num=conv_layer_num, i=0)




block_num: 2
conv_layer_num: 2
hidden_size_list: [64, 128, 256, 512, 1024, 2048, 4096]


Starting Epoch: 1...
Starting Epoch: 2...
Starting Epoch: 3...
Starting Epoch: 4...
Starting Epoch: 5...
Starting Epoch: 6...
Starting Epoch: 7...
Starting Epoch: 8...
Starting Epoch: 9...
Starting Epoch: 10...
Starting Epoch: 11...
Starting Epoch: 12...
Starting Epoch: 13...
Starting Epoch: 14...
Starting Epoch: 15...
Starting Epoch: 16...
Starting Epoch: 17...
Starting Epoch: 18...
Starting Epoch: 19...
Starting Epoch: 20...


accuracy,▁
accuracy,73.49159


In [11]:


# class SimpleFCNN(nn.Module):
#     def __init__(self, input_size=64, hidden_sizes=[512, 256, 128], num_classes=2):
#         super(SimpleFCNN, self).__init__()
        
#         # 입력 크기 계산 (RGB 이미지)
#         self.input_features = input_size * input_size * 3
        
#         # 은닉층들
#         layers = []
#         prev_size = self.input_features
        
#         for hidden_size in hidden_sizes:
#             layers.append(nn.Linear(prev_size, hidden_size))
#             layers.append(nn.ReLU())
#             layers.append(nn.Dropout(0.3))
#             prev_size = hidden_size
        
#         # 출력층
#         layers.append(nn.Linear(prev_size, num_classes))
        
#         self.network = nn.Sequential(*layers)
    
#     def forward(self, x):
#         # 이미지를 1차원으로 평탄화
#         x = x.view(x.size(0), -1)
#         return self.network(x)

In [12]:

# def process_model_1(learning_rate=0.001, image_size=64, batch_size=128, hidden_size_list=[32,64,128],  epochs=10, i=0):
#     #데이터 호출
#     wandb.init(project="cnn_test_img_265", name=f"fcnn_{i}",config={
#         "learning_rate": learning_rate, 
#         "image_size": image_size,
#         "batch_size": batch_size, 
#         "epochs": epochs,
#         "hidden_size_list": hidden_size_list,
#         })
#     device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
#     train_loader, val_loader, test_loader= data_load(image_size, batch_size)
#     hidden_size_list.reverse()
#     # model= convolution(conv1_out=hidden_size_list[0], conv2_out=hidden_size_list[1], fc1_out=hidden_size_list[2], fc2_out=2, input_size=image_size)
#     model =SimpleFCNN(input_size=image_size, hidden_sizes=hidden_size_list, num_classes=2)
#     #optimizer 선택
#     optimizer= optim.Adam(model.parameters(), lr=learning_rate)
#     criterion= nn.CrossEntropyLoss()

#     #모델 생성
#     #모델 학습
#     model= train_model(model=model,device=device, train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, epochs=epochs)
    
#     #모델평가
#     accuracy, conf_matrix= evaluate_model(model=model, device=device, test_loader=test_loader)
#     wandb.log({"accuracy": accuracy, "conf_matrix": conf_matrix})
#     wandb.finish()

In [ ]:
# for i in range(5):
#     for learning_rate in params['learning_rate']:
#        for image_size in params['image_size']:
#            for batch_size in params['batch_size']:
#                for epochs in params['epochs']:
#                    for hidden_size_list in params['hidden_sizes']:
#                        print(f'Starting process: {i}...')
#                        process_model_1(learning_rate=learning_rate, image_size=image_size, batch_size=batch_size, epochs=epochs, hidden_size_list=hidden_size_list, i=i)



